# M07 · Demostración: de índices y forma a una ejecución verificable

Este notebook implementa el caso conductor de `lesson.md`. PyTorch se usa
después de formular la relación matemática. En cada bloque seguiremos:

**formular → predecir → implementar → contrastar → explicar**.

No se estudian gradientes, autodiferenciación, pérdidas, optimizadores ni
entrenamiento; esos contenidos pertenecen a M08.



In [ ]:
import sys

import torch

torch.set_printoptions(precision=4, sci_mode=False)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Dispositivo usado: cpu")



## 1 · Registro semántico del caso conductor

Adoptamos una sola convención: las observaciones ocupan filas.

| Símbolo | Forma | Ejes | Índices |
|---|---|---|---|
| $X$ | $(B,D)$ | observación, entrada | $i,d$ |
| $W$ | $(D,H)$ | entrada, salida | $d,h$ |
| $b$ | $(H,)$ | salida | $h$ |
| $Y$ | $(B,H)$ | observación, salida | $i,h$ |

Antes de ejecutar se deriva

$$Y_{ih}=\sum_{d=1}^{D}X_{id}W_{dh}+b_h.$$

El índice $d$ se contrae; $i$ y $h$ permanecen libres. Por tanto, la forma
predicha de $Y$ es $(B,H)=(3,3)$.



In [ ]:
X = torch.tensor(
    [[1.0, 2.0], [0.0, -1.0], [3.0, 1.0]],
    dtype=torch.float32,
    device="cpu",
)
W = torch.tensor(
    [[2.0, -1.0, 0.0], [1.0, 1.0, 2.0]],
    dtype=torch.float32,
    device="cpu",
)
b = torch.tensor([1.0, 0.0, -2.0], dtype=torch.float32, device="cpu")

register = {
    "X": {"shape": (3, 2), "axes": ("observación", "entrada"), "indices": "i,d"},
    "W": {"shape": (2, 3), "axes": ("entrada", "salida"), "indices": "d,h"},
    "b": {"shape": (3,), "axes": ("salida",), "indices": "h"},
    "Y": {"shape": (3, 3), "axes": ("observación", "salida"), "indices": "i,h"},
}

assert tuple(X.shape) == register["X"]["shape"]
assert tuple(W.shape) == register["W"]["shape"]
assert tuple(b.shape) == register["b"]["shape"]
register



`shape`, `dtype` y `device` describen la representación computacional. No
almacenan el significado de “observación”, “entrada” o “salida”; ese
significado procede del contrato que acabamos de declarar.



In [ ]:
metadata = {
    "shape": tuple(X.shape),
    "ndim": X.ndim,
    "dtype": str(X.dtype),
    "device": X.device.type,
    "layout": str(X.layout),
}
print(metadata)
assert metadata["shape"] == (3, 2)
assert metadata["ndim"] == 2
assert metadata["device"] == "cpu"



## 2 · Cuatro mecanismos distintos

- Hadamard conserva los índices: $C_{ij}=A_{ij}B_{ij}$.
- El producto exterior crea el par libre: $O_{ij}=u_i v_j$.
- Una permutación reordena ejes sin combinar valores.
- Una reducción suma un índice y lo elimina del resultado.



Antes de ejecutar: Hadamard debe tener forma $(2,2)$, el exterior $(2,3)$,
la permutación $(2,4,5)$ y la reducción $(4,2)$.



In [ ]:
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
B_matrix = torch.tensor([[2.0, 0.0], [-1.0, 5.0]])
u = torch.tensor([1.0, 2.0])
v = torch.tensor([3.0, -1.0, 4.0])
S = torch.arange(4 * 5 * 2, dtype=torch.float32).reshape(4, 5, 2)

hadamard = A * B_matrix
outer = torch.outer(u, v)
permuted = S.permute(2, 0, 1)
reduced = S.sum(dim=1)

assert torch.equal(hadamard, torch.tensor([[2.0, 0.0], [-3.0, 20.0]]))
assert torch.equal(outer, torch.tensor([[3.0, -1.0, 4.0], [6.0, -2.0, 8.0]]))
assert tuple(permuted.shape) == (2, 4, 5)
assert tuple(reduced.shape) == (4, 2)

print("Hadamard:", tuple(hadamard.shape), "índices i,j libres")
print("Exterior:", tuple(outer.shape), "índices i,j creados")
print("Permutación:", tuple(S.shape), "→", tuple(permuted.shape))
print("Reducción en t:", tuple(S.shape), "→", tuple(reduced.shape))



In [ ]:
observed_mechanism_shapes = {
    "hadamard": tuple(hadamard.shape),
    "exterior": tuple(outer.shape),
    "permutación": tuple(permuted.shape),
    "reducción": tuple(reduced.shape),
}
assert observed_mechanism_shapes == {
    "hadamard": (2, 2),
    "exterior": (2, 3),
    "permutación": (2, 4, 5),
    "reducción": (4, 2),
}



Las formas coinciden, pero la evidencia central es el patrón de índices:
conservar, crear, reordenar y eliminar no son la misma operación.



## 3 · Broadcasting: compatibilidad dimensional y validez semántica

Para $Z\in\mathbb{R}^{B\times H}$, un sesgo por salida $b_h$ se alinea con el
último eje. Un ajuste por observación $r_i$ debe representarse como $(B,1)$.
Cuando $B=H$, una escritura incorrecta puede ejecutar sin excepción.



Si $r$ representa observaciones, `r[:, None]` debe producir filas constantes;
`r` sin eje explícito producirá columnas desplazadas por $r_h$.



In [ ]:
Z = torch.zeros((3, 3))
r = torch.tensor([10.0, 20.0, 30.0])

wrong = Z + r           # calcula Z_ih + r_h
right = Z + r[:, None]  # calcula Z_ih + r_i

print("misma forma externa:", tuple(wrong.shape), tuple(right.shape))
print("wrong[0] =", wrong[0].tolist())
print("right[0] =", right[0].tolist())

assert tuple(wrong.shape) == tuple(right.shape) == (3, 3)
assert torch.equal(wrong[0], torch.tensor([10.0, 20.0, 30.0]))
assert torch.equal(right[0], torch.tensor([10.0, 10.0, 10.0]))
assert not torch.equal(wrong, right)



In [ ]:
assert torch.equal(right[:, 0], r)
assert torch.equal(wrong[0], r)
print("La orientación observada distingue r_i de r_h.")



La API aplicó correctamente su regla de alineación. El error está en haber
proporcionado una forma compatible con un significado distinto del declarado.



## 4 · Transformación afín: componentes, matriz y código

La relación por componentes

$$Y_{ih}=\sum_d X_{id}W_{dh}+b_h$$

equivale, bajo la convención de filas, a

$$Y=XW+\mathbf{1}_B b^{\mathsf T}.$$

Ahora sí traducimos ambas escrituras a PyTorch.



El cálculo manual predice $XW\in\mathbb{R}^{3\times3}$ y
$Y\in\mathbb{R}^{3\times3}$; agregar $b_h$ no cambia el eje de observación.



In [ ]:
expected_XW = torch.tensor(
    [[4.0, 1.0, 4.0], [-1.0, -1.0, -2.0], [7.0, -2.0, 2.0]]
)
expected_Y = torch.tensor(
    [[5.0, 1.0, 2.0], [0.0, -1.0, -4.0], [8.0, -2.0, 0.0]]
)

XW = X @ W
ones = torch.ones((X.shape[0], 1), dtype=X.dtype, device=X.device)
explicit = XW + ones @ b[None, :]
broadcasted = XW + b

assert tuple(broadcasted.shape) == register["Y"]["shape"]
assert torch.equal(XW, expected_XW)
assert torch.equal(explicit, expected_Y)
assert torch.equal(broadcasted, expected_Y)
assert torch.equal(explicit, broadcasted)

print("XW =\n", XW)
print("Y =\n", broadcasted)



In [ ]:
assert tuple(XW.shape) == (3, 3)
assert tuple(broadcasted.shape) == (3, 3)
assert torch.equal(broadcasted[0], torch.tensor([5.0, 1.0, 2.0]))



La igualdad numérica confirma una predicción previa. No determina por sí sola
que $b$ signifique desplazamiento por salida; esa interpretación fue declarada
antes del cálculo.



## 5 · `matmul` como contracción con ejes de contexto

Si $S\in\mathbb{R}^{B\times T\times D}$ y
$W_{seq}\in\mathbb{R}^{D\times H}$, entonces

$$U_{ith}=\sum_d S_{itd}(W_{seq})_{dh},$$

por lo que la forma predicha es $(B,T,H)$. Los ejes $i,t$ sobreviven y $d$ se
contrae.



La contracción elimina $d$ y conserva los dos ejes de contexto $i,t$; por
tanto la forma esperada es $(4,5,3)$.



In [ ]:
B_size, T_size, D_size, H_size = 4, 5, 2, 3
S_sequence = torch.arange(
    B_size * T_size * D_size, dtype=torch.float32
).reshape(B_size, T_size, D_size)
W_sequence = torch.tensor([[1.0, 0.0, -1.0], [0.5, 2.0, 1.0]])

predicted_shape = (B_size, T_size, H_size)
U = torch.matmul(S_sequence, W_sequence)

assert tuple(U.shape) == predicted_shape
assert torch.equal(U[0, 0], S_sequence[0, 0] @ W_sequence)
print("forma predicha y observada:", predicted_shape, tuple(U.shape))



In [ ]:
manual_first = torch.stack(
    [S_sequence[0, 0] @ W_sequence[:, h] for h in range(H_size)]
)
assert torch.equal(U[0, 0], manual_first)



`matmul` actúa sobre los dos últimos ejes; los ejes anteriores sobreviven
porque no participan en la suma sobre $d$.



## 6 · Representación: `dtype`, dispositivo, `reshape`, `view` y `permute`

Estas operaciones describen o reorganizan almacenamiento. No crean por sí
mismas una nueva relación matemática ni preservan automáticamente la semántica.



La conversión cambiará `dtype`, la permutación cambiará la forma y los
`reshape` conservarán 12 elementos. Ninguna operación asignará semántica.



In [ ]:
T = torch.tensor([[1, 2], [3, 4]], dtype=torch.int64, device="cpu")
T_float = T.to(dtype=torch.float32)
base = torch.arange(12).reshape(3, 4)
flat_view = base.view(12)
flat_reshape = base.reshape(12)
base_permuted = base.permute(1, 0)

assert T.dtype == torch.int64
assert T_float.dtype == torch.float32
assert T.device.type == "cpu"
assert tuple(flat_view.shape) == tuple(flat_reshape.shape) == (12,)
assert tuple(base_permuted.shape) == (4, 3)
assert base.numel() == base_permuted.numel() == 12
assert base.is_contiguous()
assert not base_permuted.is_contiguous()

print("base.stride() =", base.stride())
print("permuted.stride() =", base_permuted.stride())



In [ ]:
assert flat_view.tolist() == list(range(12))
assert flat_reshape.tolist() == list(range(12))
assert base_permuted[0].tolist() == [0, 4, 8]



`view` exige una organización compatible; `reshape` puede devolver vista o
copia. Por ello el código no debe basar su significado en que `reshape`
comparta almacenamiento. El protocolo seguro registra forma y semántica antes
y después, además del número de elementos.



## 7 · Clínica de error semántico

El nombre `offset_by_observation` declara $offset_i$, pero `scores + offset`
alinea el vector con el último eje y calcula $offset_h$.



La intención predice filas desplazadas por $10$, $20$ y $30$; el candidato
sin `[:, None]` producirá columnas desplazadas por esos valores.



In [ ]:
scores = torch.tensor(
    [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]
)
offset_by_observation = torch.tensor([10.0, 20.0, 30.0])

candidate = scores + offset_by_observation
corrected = scores + offset_by_observation[:, None]
expected = torch.tensor(
    [[11.0, 12.0, 13.0], [24.0, 25.0, 26.0], [37.0, 38.0, 39.0]]
)

assert tuple(candidate.shape) == tuple(corrected.shape) == (3, 3)
assert torch.equal(corrected, expected)
assert not torch.equal(candidate, corrected)

print("candidate =\n", candidate)
print("corrected =\n", corrected)



In [ ]:
assert torch.equal(corrected[:, 0], torch.tensor([11.0, 24.0, 37.0]))
assert torch.equal(candidate[0], torch.tensor([11.0, 22.0, 33.0]))



Dos aserciones complementarias son necesarias: una confirma la relación
declarada y otra detecta el candidato que ejecuta sobre el eje equivocado.

El contraste conceptual con TensorFlow no abre una segunda ruta: la misma
relación se escribiría con `tf.linalg.matmul(X_tf, W_tf) + b_tf`. La
formulación matemática y la predicción de forma permanecen iguales.



## 8 · Cierre

La cadena verificada fue:

$$
X_{id},W_{dh},b_h
\rightarrow d\text{ contraído; }i,h\text{ libres}
\rightarrow Y_{ih}=\sum_dX_{id}W_{dh}+b_h
\rightarrow Y=XW+\mathbf{1}_Bb^{\mathsf T}
\rightarrow (B,H)
\rightarrow \texttt{X @ W + b}.
$$

La ejecución cierra la comprobación; la justificación de cantidades, ejes e
índices cierra el razonamiento.
